In [2]:
import torch 
import pandas as pd 
import matplotlib.pyplot as plt
import time 
import numpy as np 
from torchvision import datasets 
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F 
import torch


In [3]:

RANDOM_SEED = 1
BATCH_SIZE = 100
NUM_EPOCHS = 100
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [5]:
# Load the data 
# transforms.ToTensor() scales input images to 0 to 1 range
train_dataset = datasets.MNIST(root = 'data', 
                               train = True, 
                               transform = transforms.ToTensor(), 
                               download = True)

test_dataset = datasets.MNIST(root = 'data', 
                              train = False, 
                              transform = transforms.ToTensor())



train_loader = DataLoader(dataset=train_dataset, 
                          batch_size=BATCH_SIZE, 
                          shuffle=True)

test_loader = DataLoader(dataset=test_dataset, 
                         batch_size=BATCH_SIZE, 
                         shuffle=False)

100.0%
100.0%
100.0%
100.0%


In [7]:
# Checking the dataset
for images, labels in train_loader:  
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break


Image batch dimensions: torch.Size([100, 1, 28, 28])
Image label dimensions: torch.Size([100])


# Model

In [6]:
class MLP(torch.nn.Module):

  def __init__(self, num_features, num_hidden, num_classes):
    super().__init__()

    self.num_classes = num_classes

    # 1st hidden layer
    self.linear_1 = torch.nn.Linear(num_features, num_hidden)
    self.linear_1.weight.detach().normal_(0.0, 0.1)
    self.linear_1.bias.detach().zero_()

    ## output layer
    self.linear_out = torch.nn.Linear(num_hidden, num_classes)
    self.linear_out.weight.detach().normal_(0.0, 0.1)
    self.linear_out.bias.detach().zero_()


  def forward(self, x):
    out = self.linear_1(x)
    out = torch.sigmoid(out)
    logits = self.linear_out
    # probas = torch.softmax(logits, dim = 1)
    return logits #, probas
  
  

In [8]:
# initialise the model

torch.manual_seed(RANDOM_SEED)
model = MLP(num_features=28*28, 
            num_hidden = 100, 
            num_classes = 10)

model = model.to(DEVICE)
optimizer= torch.optim.SGD(model.parameters(), lr = 0.1)


In [9]:
# train


def compute_loss(net, data_loader):
  curr_loss = 0 

  with torch.no_grad():
    for cnt, (features, targets) in enumerate(data_loader):
      features = features.view(-1, 28*28).to(DEVICE)
      targets.to(DEVICE)
      logits = net(features)
      loss = F.cross_entropy(logits, targets)
      curr_loss += loss

    return float(curr_loss)/cnt


In [12]:
start_time = time.time()
minibatch_cost = []
epoch_cost = []
for epoch in range(NUM_EPOCHS):
    model.train()
    for batch_idx, (features, targets) in enumerate(train_loader):
        
        features = features.view(-1, 28*28).to(DEVICE)
        targets = targets.to(DEVICE)
            
        ### FORWARD AND BACK PROP
        logits = model(features)
        
        cost = F.cross_entropy(logits, targets)
        optimizer.zero_grad()
        
        cost.backward()
       
        ### UPDATE MODEL PARAMETERS
        optimizer.step()
        
        ### LOGGING
        minibatch_cost.append(cost.item())
        if not batch_idx % 50:
            print ('Epoch: %03d/%03d | Batch %03d/%03d | Cost: %.4f' 
                   %(epoch+1, NUM_EPOCHS, batch_idx, 
                     len(train_loader), cost.item()))
        
    cost = compute_loss(model, train_loader)
    epoch_cost.append(cost)
    print('Epoch: %03d/%03d Train Cost: %.4f' % (
            epoch+1, NUM_EPOCHS, cost))
    print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    
print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))


TypeError: cross_entropy_loss(): argument 'input' (position 1) must be Tensor, not Linear